# Week 3: FlashAttention-Style Tiled Transformation

This notebook extends Week 2 by implementing a tiled attention transformation inspired by FlashAttention-style execution.

In standard scaled dot-product attention, the full attention score matrix is materialized in memory, resulting in quadratic memory complexity with respect to sequence length. This becomes a major bottleneck for long-context workloads and large-scale inference systems.

To address this issue, this notebook introduces a block-wise tiled computation strategy. Instead of computing the full attention matrix at once, attention is computed incrementally over smaller query and key-value tiles. This reduces the size of intermediate score blocks and provides the foundation for more memory-efficient execution.

The goals of this notebook are:

1. Reuse the baseline attention and DSL interface from earlier weeks.
2. Implement a tiled attention transformation.
3. Integrate the tiled transformation into the DSL lowering path.
4. Verify correctness against the baseline implementation.
5. Compare runtime and memory behavior for baseline and tiled attention.

This implementation is inspired by FlashAttention-style block-wise execution, but it is still a high-level Python/PyTorch prototype rather than a fused low-level kernel.

## Week 3 Objectives

This week focuses on the first real transformation step in the project. In Week 2, the DSL lowered attention specifications directly to baseline PyTorch implementations. In Week 3, the lowering path is extended so that selected attention workloads are translated into tiled execution.

The main purpose of this notebook is to demonstrate that attention can be computed in a block-wise manner without materializing the full score matrix at once, while still preserving correctness relative to the standard baseline implementation.

In [ ]:
import math
import time
import torch
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

Using device: cuda
GPU: Tesla T4


## Baseline Attention Functions

The following baseline functions are reused from earlier weeks and serve as the reference implementations for correctness checks.

In [ ]:
def baseline_attention(Q, K, V, causal=False):
    """
    Standard scaled dot-product attention.
    Supports optional causal masking.
    """
    d_k = Q.size(-1)
    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)

    if causal:
        seq_len_q = Q.size(-2)
        seq_len_k = K.size(-2)
        mask = torch.triu(
            torch.ones(seq_len_q, seq_len_k, device=Q.device, dtype=torch.bool),
            diagonal=1
        )
        scores = scores.masked_fill(mask, float("-inf"))

    weights = torch.softmax(scores, dim=-1)
    return torch.matmul(weights, V)

In [ ]:
def streaming_attention(Q_seq, K_seq, V_seq):
    """
    Simple streaming-style reference attention.
    At time step t, attention is computed only over the prefix up to t.
    """
    outputs = []

    for t in range(Q_seq.size(1)):
        Q_t = Q_seq[:, t:t+1, :]
        K_t = K_seq[:, :t+1, :]
        V_t = V_seq[:, :t+1, :]
        out_t = baseline_attention(Q_t, K_t, V_t, causal=False)
        outputs.append(out_t)

    return torch.cat(outputs, dim=1)

## DSL Interface

The same Python-embedded DSL from Week 2 is reused here. The main difference is that the `tile_size` field now affects execution rather than serving only as metadata.

In [ ]:
class AttentionSpec:
    """
    Lightweight DSL object for declarative attention specification.
    """
    def __init__(
        self,
        Q,
        K,
        V,
        name="attention_op",
        scenario="generic",
        causal=False,
        tile_size=None,
        streaming=False,
        variable_length=False,
        cache_kv=False
    ):
        self.Q = Q
        self.K = K
        self.V = V
        self.name = name
        self.scenario = scenario
        self.causal = causal
        self.tile_size = tile_size
        self.streaming = streaming
        self.variable_length = variable_length
        self.cache_kv = cache_kv

    def summary(self):
        return {
            "name": self.name,
            "scenario": self.scenario,
            "causal": self.causal,
            "tile_size": self.tile_size,
            "streaming": self.streaming,
            "variable_length": self.variable_length,
            "cache_kv": self.cache_kv,
            "Q_shape": tuple(self.Q.shape),
            "K_shape": tuple(self.K.shape),
            "V_shape": tuple(self.V.shape),
        }

## Tiled Attention Transformation

The following function implements a tiled attention computation. It processes the query sequence in blocks and computes attention scores against key-value blocks instead of forming the full score matrix at once.

This implementation is meant to simulate the core idea of FlashAttention-style execution at a high level. It is not yet a fused kernel and still uses standard PyTorch operations, but it demonstrates the transformation from full attention to block-wise execution.

In [ ]:
def tiled_attention(Q, K, V, tile_size=64, causal=False):
    """
    High-level tiled attention implementation.

    This function processes query blocks against key-value blocks
    and avoids explicitly computing the full attention score matrix
    in one step.

    Note:
    This is a Python/PyTorch prototype inspired by FlashAttention-style
    tiling. It is not a fused low-level implementation.
    """
    B, N, D = Q.shape
    output = torch.zeros_like(Q)

    for i in range(0, N, tile_size):
        i_end = min(i + tile_size, N)
        Q_tile = Q[:, i:i_end, :]

        score_tiles = []
        value_tiles = []

        for j in range(0, N, tile_size):
            j_end = min(j + tile_size, N)
            K_tile = K[:, j:j_end, :]
            V_tile = V[:, j:j_end, :]

            scores = torch.matmul(Q_tile, K_tile.transpose(-2, -1)) / math.sqrt(D)

            if causal:
                q_idx = torch.arange(i, i_end, device=Q.device).unsqueeze(1)
                k_idx = torch.arange(j, j_end, device=Q.device).unsqueeze(0)
                causal_mask = k_idx > q_idx
                scores = scores.masked_fill(causal_mask.unsqueeze(0), float("-inf"))

            score_tiles.append(scores)
            value_tiles.append(V_tile)

        full_tile_scores = torch.cat(score_tiles, dim=-1)
        weights = torch.softmax(full_tile_scores, dim=-1)

        tile_output = torch.zeros(B, i_end - i, D, device=Q.device, dtype=Q.dtype)

        start = 0
        for V_tile in value_tiles:
            end = start + V_tile.size(1)
            tile_weights = weights[:, :, start:end]
            tile_output += torch.matmul(tile_weights, V_tile)
            start = end

        output[:, i:i_end, :] = tile_output

    return output

## Quick Sanity Check for Tiled Attention

Before integrating tiled attention into the DSL lowering path, we verify that it matches the standard baseline implementation on a small example.

In [ ]:
B, N, D = 2, 64, 32

Q = torch.randn(B, N, D, device=device)
K = torch.randn(B, N, D, device=device)
V = torch.randn(B, N, D, device=device)

baseline_out = baseline_attention(Q, K, V, causal=False)
tiled_out = tiled_attention(Q, K, V, tile_size=16, causal=False)

max_diff = (baseline_out - tiled_out).abs().max().item()
print("Maximum absolute difference:", max_diff)
print("Outputs match:", torch.allclose(baseline_out, tiled_out, atol=1e-6))

Maximum absolute difference: 3.5762786865234375e-07
Outputs match: True


## Causal Attention Sanity Check

Because long-context autoregressive workloads often require causal masking, we also verify that the tiled implementation matches the causal baseline attention result.

In [ ]:
baseline_causal_out = baseline_attention(Q, K, V, causal=True)
tiled_causal_out = tiled_attention(Q, K, V, tile_size=16, causal=True)

causal_max_diff = (baseline_causal_out - tiled_causal_out).abs().max().item()
print("Maximum absolute difference (causal):", causal_max_diff)
print("Outputs match:", torch.allclose(baseline_causal_out, tiled_causal_out, atol=1e-6))

Maximum absolute difference (causal): 4.76837158203125e-07
Outputs match: True


## Updated DSL Lowering

In Week 2, the DSL lowered attention specifications directly to baseline attention. In this notebook, the lowering stage is extended so that specifications with a tile size are routed through the tiled attention implementation.

Streaming workloads still use the streaming reference function for now.

In [ ]:
def lower_attention(spec):
    """
    Lower a DSL attention specification to an executable implementation.
    Priority:
    1. Streaming reference implementation
    2. Tiled attention if tile_size is provided
    3. Standard baseline attention
    """
    if spec.streaming:
        return streaming_attention(spec.Q, spec.K, spec.V)

    if spec.tile_size is not None:
        return tiled_attention(
            spec.Q,
            spec.K,
            spec.V,
            tile_size=spec.tile_size,
            causal=spec.causal
        )

    return baseline_attention(spec.Q, spec.K, spec.V, causal=spec.causal)

## Scenario 1: Multi-Tenant Serving

This scenario represents a request in a multi-tenant environment where sequence lengths may vary across users. In this prototype, tiled execution is applied through the DSL.

In [ ]:
Q_mt = torch.randn(1, 128, 64, device=device)
K_mt = torch.randn(1, 128, 64, device=device)
V_mt = torch.randn(1, 128, 64, device=device)

multitenant_spec = AttentionSpec(
    Q=Q_mt,
    K=K_mt,
    V=V_mt,
    name="multitenant_request",
    scenario="multi_tenant",
    causal=False,
    tile_size=32,
    streaming=False,
    variable_length=True,
    cache_kv=False
)

pd.DataFrame([multitenant_spec.summary()])

,name,scenario,causal,tile_size,streaming,variable_length,cache_kv,Q_shape,K_shape,V_shape
0,multitenant_request,multi_tenant,False,32,False,True,False,"(1, 128, 64)","(1, 128, 64)","(1, 128, 64)"


In [ ]:
mt_dsl_output = lower_attention(multitenant_spec)
mt_ref_output = baseline_attention(Q_mt, K_mt, V_mt, causal=False)

mt_max_diff = (mt_dsl_output - mt_ref_output).abs().max().item()
print("Multi-tenant max difference:", mt_max_diff)
print("Outputs match:", torch.allclose(mt_dsl_output, mt_ref_output, atol=1e-6))

Multi-tenant max difference: 7.152557373046875e-07
Outputs match: True


## Scenario 2: Streaming Inference

Streaming inference is still lowered to the streaming reference implementation. This keeps the notebook focused on the tiled transformation while preserving support for scenario-specific lowering behavior.

In [ ]:
Q_stream = torch.randn(1, 32, 64, device=device)
K_stream = torch.randn(1, 32, 64, device=device)
V_stream = torch.randn(1, 32, 64, device=device)

streaming_spec = AttentionSpec(
    Q=Q_stream,
    K=K_stream,
    V=V_stream,
    name="streaming_request",
    scenario="streaming",
    causal=False,
    tile_size=16,
    streaming=True,
    variable_length=False,
    cache_kv=True
)

pd.DataFrame([streaming_spec.summary()])

,name,scenario,causal,tile_size,streaming,variable_length,cache_kv,Q_shape,K_shape,V_shape
0,streaming_request,streaming,False,16,True,False,True,"(1, 32, 64)","(1, 32, 64)","(1, 32, 64)"


In [ ]:
stream_dsl_output = lower_attention(streaming_spec)
stream_ref_output = streaming_attention(Q_stream, K_stream, V_stream)

stream_max_diff = (stream_dsl_output - stream_ref_output).abs().max().item()
print("Streaming max difference:", stream_max_diff)
print("Outputs match:", torch.allclose(stream_dsl_output, stream_ref_output, atol=1e-6))

Streaming max difference: 0.0
Outputs match: True


## Scenario 3: Long-Context Attention

Long-context workloads are especially relevant for tiled attention because they stress the quadratic memory behavior of standard attention. This scenario applies causal masking together with tiled execution.

In [ ]:
Q_long = torch.randn(1, 256, 64, device=device)
K_long = torch.randn(1, 256, 64, device=device)
V_long = torch.randn(1, 256, 64, device=device)

long_context_spec = AttentionSpec(
    Q=Q_long,
    K=K_long,
    V=V_long,
    name="long_context_request",
    scenario="long_context",
    causal=True,
    tile_size=64,
    streaming=False,
    variable_length=False,
    cache_kv=False
)

pd.DataFrame([long_context_spec.summary()])

,name,scenario,causal,tile_size,streaming,variable_length,cache_kv,Q_shape,K_shape,V_shape
0,long_context_request,long_context,True,64,False,False,False,"(1, 256, 64)","(1, 256, 64)","(1, 256, 64)"


In [ ]:
long_dsl_output = lower_attention(long_context_spec)
long_ref_output = baseline_attention(Q_long, K_long, V_long, causal=True)

long_max_diff = (long_dsl_output - long_ref_output).abs().max().item()
print("Long-context max difference:", long_max_diff)
print("Outputs match:", torch.allclose(long_dsl_output, long_ref_output, atol=1e-6))

Long-context max difference: 5.364418029785156e-07
Outputs match: True


## Runtime Benchmarking

The next step is to compare runtime behavior between the standard baseline implementation and the tiled implementation. Because the tiled version is written with Python loops rather than fused kernels, it may not be faster in wall-clock time. However, it still demonstrates the structural transformation toward memory-aware execution.

In [ ]:
def benchmark(fn, *args, repeats=10, **kwargs):
    times = []

    for _ in range(repeats):
        if device == "cuda":
            torch.cuda.synchronize()

        start = time.time()
        fn(*args, **kwargs)

        if device == "cuda":
            torch.cuda.synchronize()

        end = time.time()
        times.append(end - start)

    return sum(times) / len(times)

In [ ]:
Q_bench = torch.randn(1, 256, 64, device=device)
K_bench = torch.randn(1, 256, 64, device=device)
V_bench = torch.randn(1, 256, 64, device=device)

baseline_time = benchmark(
    baseline_attention,
    Q_bench, K_bench, V_bench,
    repeats=10,
    causal=True
)

tiled_time = benchmark(
    tiled_attention,
    Q_bench, K_bench, V_bench,
    repeats=10,
    tile_size=64,
    causal=True
)

timing_df = pd.DataFrame([
    {"implementation": "baseline_attention", "avg_time_sec": baseline_time},
    {"implementation": "tiled_attention", "avg_time_sec": tiled_time}
])

timing_df

,implementation,avg_time_sec
0,baseline_attention,0.000548
1,tiled_attention,0.003598


## Approximate Intermediate Memory Comparison

A simple way to illustrate the memory difference is to compare the size of the largest score tensor used by each implementation.

For baseline attention, the full score matrix has shape `(B, N, N)`.
For tiled attention, each score block has shape `(B, tile_size, tile_size)` for each local computation block, although the current high-level prototype still concatenates score tiles within each query block.

This comparison is approximate, but it helps illustrate how tiled execution reduces the size of local intermediate computations.

In [ ]:
def tensor_memory_mb(shape, dtype=torch.float32):
    bytes_per_element = torch.tensor([], dtype=dtype).element_size()
    num_elements = 1
    for dim in shape:
        num_elements *= dim
    return (num_elements * bytes_per_element) / (1024 ** 2)

B, N, D = 1, 256, 64
tile_size = 64

baseline_score_mb = tensor_memory_mb((B, N, N))
tile_score_mb = tensor_memory_mb((B, tile_size, tile_size))

memory_df = pd.DataFrame([
    {
        "tensor_type": "baseline full score matrix",
        "shape": (B, N, N),
        "approx_memory_mb": baseline_score_mb
    },
    {
        "tensor_type": "tiled local score block",
        "shape": (B, tile_size, tile_size),
        "approx_memory_mb": tile_score_mb
    }
])

memory_df

,tensor_type,shape,approx_memory_mb
0,baseline full score matrix,"(1, 256, 256)",0.250000
1,tiled local score block,"(1, 64, 64)",0.015625


## Scenario Summary Table

The following table summarizes the correctness results for each workload type tested in this notebook.

In [ ]:
results_df = pd.DataFrame([
    {
        "scenario": "multi_tenant",
        "max_abs_diff": mt_max_diff,
        "matches_reference": torch.allclose(mt_dsl_output, mt_ref_output, atol=1e-6)
    },
    {
        "scenario": "streaming",
        "max_abs_diff": stream_max_diff,
        "matches_reference": torch.allclose(stream_dsl_output, stream_ref_output, atol=1e-6)
    },
    {
        "scenario": "long_context",
        "max_abs_diff": long_max_diff,
        "matches_reference": torch.allclose(long_dsl_output, long_ref_output, atol=1e-6)
    }
])

results_df

,scenario,max_abs_diff,matches_reference
0,multi_tenant,7.152557e-07,True
1,streaming,0.000000e+00,True
2,long_context,5.364418e-07,True


## Week 3 Observations

This week introduced the first transformation-oriented stage of the project by replacing direct baseline lowering with a tiled attention implementation. The tiled computation processes attention in blocks rather than forming the full attention score matrix in a single step, which better reflects the structure of memory-efficient attention methods such as FlashAttention.

The correctness results show that the tiled implementation matches the reference baseline outputs for the tested scenarios, including causal long-context attention. This is important because it demonstrates that the transformation preserves the semantics of standard attention.

The runtime comparison may show that the tiled implementation is not faster than the baseline in this prototype. This is expected because the current implementation uses Python loops and standard PyTorch operations rather than fused low-level kernels. The primary contribution of this week is therefore the transformation structure itself, not raw performance improvement.

The memory comparison illustrates that local score blocks are substantially smaller than the full score matrix used by standard attention. This supports the core motivation for tiled execution and establishes the basis for more advanced streaming and memory-optimized execution in later weeks.

## Week 3 Deliverables Completed

- Reused the Week 2 DSL interface for attention specification
- Implemented a tiled attention transformation inspired by FlashAttention-style execution
- Extended DSL lowering to route selected workloads to tiled execution
- Verified correctness against the baseline implementation
- Compared baseline and tiled runtime behavior
- Illustrated the reduction in local intermediate score block size